# Proyecto - Asistente de IA para aerolínea

Ahora vamos a reunir lo que hemos aprendido para crear un asistente de IA de atención al cliente para una aerolínea

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Inicialización

load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"La clave de API de Google existe y comienza con {google_api_key[:8]}")
else:
    print("La clave de API de Google no está configurada")

MODEL = "gemini-3.1-flash-lite"
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# Como alternativa, si quieres usar Ollama en lugar de Gemini
# Verifica que Ollama esté corriendo localmente (ver el ejercicio de week1/day2) y luego descomenta estas 2 líneas
# MODEL = "llama3.2"
# gemini = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

La clave de API de Google existe y comienza con AQ.Ab8RN


In [3]:
system_message = """
Eres un asistente útil para una aerolínea llamada FlightAI.
Da respuestas breves y amables, de no más de 1 oración.
Sé siempre preciso. Si no sabes la respuesta, dilo.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Herramientas

Las herramientas son una función increíblemente poderosa que ofrecen los LLMs de frontera.

Con las herramientas, puedes escribir una función y hacer que el LLM la llame como parte de su respuesta.

¿Suena casi espeluznante... le estamos dando el poder de ejecutar código en nuestra máquina?

Bueno, más o menos.

In [5]:
# Empecemos creando una función útil
import unicodedata

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

# Nombres alternativos (español, etc.) → clave de ticket_prices
city_aliases = {"londres": "london", "tokio": "tokyo", "parís": "paris", "berlín": "berlin"}

def normalize_city(city):
    # minúsculas, sin espacios sobrantes y sin tildes: "  París " → "paris"
    city = unicodedata.normalize("NFKD", city.strip().lower())
    return "".join(c for c in city if not unicodedata.combining(c))

def get_ticket_price(destination_city):
    print(f"Herramienta llamada para la ciudad {destination_city}")
    key = normalize_city(destination_city)
    key = city_aliases.get(key, key)
    price = ticket_prices.get(key, "Precio de boleto desconocido")
    return f"El precio de un boleto a {destination_city} es {price}"

In [6]:
get_ticket_price("London")

Herramienta llamada para la ciudad London


'El precio de un boleto a London es $799'

In [ ]:
# Hay una estructura de diccionario particular que se requiere para describir nuestra función:

price_function = {
    "name": "get_ticket_price",
    "description": "Obtiene el precio de un boleto de ida y vuelta a la ciudad de destino.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "La ciudad a la que el cliente quiere viajar",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [8]:
# Y esto se incluye en una lista de herramientas:

tools = [{"type": "function", "function": price_function}]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Obtiene el precio de un boleto de ida y vuelta a la ciudad de destino.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'La ciudad a la que el cliente quiere viajar'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Haciendo que Gemini use nuestra herramienta

Hay algunos detalles delicados para permitir que Gemini "llame a nuestra herramienta"

Lo que hacemos en realidad es darle al LLM la oportunidad de informarnos que quiere que ejecutemos la herramienta.

Así se ve la nueva función de chat:

In [10]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)   # ← NUEVO: tools

    if response.choices[0].finish_reason == "tool_calls":          # ← el modelo NO respondió: pidió una herramienta
        assistant_message = response.choices[0].message            #   su petición
        tool_message = handle_tool_call(assistant_message)         #   tú ejecutas (la función va abajo)
        messages.append(assistant_message)                         #   1º guardas la petición...
        messages.append(tool_message)                              #   2º ...y luego el resultado
        response = gemini.chat.completions.create(model=MODEL, messages=messages)   # 2ª llamada

    return response.choices[0].message.content

In [11]:
# Tenemos que escribir esa función handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]                               # la PRIMERA petición
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)        # llega como texto JSON → diccionario
        city = arguments.get("destination_city")
        price_details = get_ticket_price(city)                      # ← LA ejecución real
        return {"role": "tool", "content": price_details, "tool_call_id": tool_call.id}

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Hagamos un par de mejoras

Manejar múltiples llamadas a herramientas en una sola respuesta

Manejar múltiples llamadas a herramientas una tras otra

In [13]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [14]:
def handle_tool_calls(message):                       # ahora en plural
    responses = []
    for tool_call in message.tool_calls:              # ← un for en vez de [0]
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            responses.append({"role": "tool",
                              "content": get_ticket_price(city),
                              "tool_call_id": tool_call.id})
    return responses

In [15]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [17]:
import sqlite3


In [18]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [19]:
def get_ticket_price(city):
    print(f"HERRAMIENTA DE BASE DE DATOS LLAMADA: Obteniendo precio para {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"El precio del boleto a {city} es ${result[0]}" if result else "No hay datos de precio disponibles para esta ciudad"

In [20]:
get_ticket_price("Tokyo")

HERRAMIENTA DE BASE DE DATOS LLAMADA: Obteniendo precio para Tokyo


'El precio del boleto a Tokyo es $1500.0'

In [21]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
    return f"Precio para {city} establecido en ${price}"    # ← AQUÍ (4 espacios, igual que el `with`)

In [22]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [23]:
get_ticket_price("Tokyo")

HERRAMIENTA DE BASE DE DATOS LLAMADA: Obteniendo precio para Tokyo


'El precio del boleto a Tokyo es $1420.0'

In [24]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Ejercicio

¡Agrega una herramienta para establecer el precio de un boleto!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Aplicaciones de negocio</h2>
            <span style="color:#181;">¡Ojalá esto apenas necesite mencionarse! Ahora tienes la capacidad de dar acciones a tus LLMs. Este asistente de aerolínea ya puede hacer más que responder preguntas: ¡podría interactuar con APIs de reservas para hacer reservas!</span>
        </td>
    </tr>
</table

In [25]:
set_price_function = {
    "name": "set_ticket_price",
    "description": "Establece o actualiza el precio de un boleto de ida y vuelta a la ciudad de destino.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {"type": "string", "description": "La ciudad de destino"},
            "price": {"type": "number", "description": "El nuevo precio del boleto en dólares"},
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False,
    },
}
tools.append({"type": "function", "function": set_price_function})

In [26]:
registry = {
    "get_ticket_price": lambda arguments: get_ticket_price(arguments["destination_city"]),
    "set_ticket_price": lambda arguments: set_ticket_price(arguments["destination_city"], arguments["price"]),
}

def run_tool(tool_call):
    function = registry.get(tool_call.function.name)
    if function is None:
        return f"Error: la herramienta '{tool_call.function.name}' no existe."
    try:
        arguments = json.loads(tool_call.function.arguments)
        return str(function(arguments))
    except (json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
        return f"Error en los argumentos: {error}"

def handle_tool_calls(message):
    return [{"role": "tool", "tool_call_id": tc.id, "content": run_tool(tc)} for tc in message.tool_calls]

In [27]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


HERRAMIENTA DE BASE DE DATOS LLAMADA: Obteniendo precio para Tokyo
